# DS 310 Project 1: Regression
**Team 9 | Michael Shepherd | mrs7065@psu.edu | shephmic77**

My final model is a blend of regression models with small prediction changes. It scored **2930.80 MSE** on the public leaderboard. The screenshot in my report shows Team 9 in first place at that time. The private result was not available.

This notebook fits the models from the competition data and recreates the submitted predictions. It does not need the old experiment files.


## 1. Load the data
Keep the supplied data folder beside this notebook. Run the cells in order. The sample submission provides the row IDs, not target labels.

The fitted model includes 50 PCA and forest fits, so the run can take a few minutes. The package versions are listed below.


In [1]:
from pathlib import Path
import os, warnings, hashlib
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler, MinMaxScaler, SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.linear_model import (Ridge, RidgeCV, LinearRegression, Lasso, ElasticNet,
    BayesianRidge, ARDRegression, HuberRegressor, GammaRegressor)
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from threadpoolctl import threadpool_limits
warnings.filterwarnings('once')
pd.set_option('display.max_rows', 30)


In [2]:
TESTED_VERSIONS = {'python': '3.12', 'numpy': '2.3.5', 'pandas': '3.0.1', 'scipy': '1.18.1', 'scikit_learn': '1.9.1'}
print('Tested environment:', TESTED_VERSIONS)
required = ['x_train.csv', 'y_train.csv', 'x_test.csv', 'sampleSubmission.csv']
folders = [Path('.'), Path('data'), Path('ds-310-fall-2026-project-1'),
           Path('../data'), Path('../ds-310-fall-2026-project-1')]
DATA = next((p for p in folders if all((p / f).is_file() for f in required)), None)
if DATA is None:
    raise FileNotFoundError('Place all four competition CSV files in one of the data folders above.')
train = pd.read_csv(DATA / 'x_train.csv')
test = pd.read_csv(DATA / 'x_test.csv')
target_frame = pd.read_csv(DATA / 'y_train.csv')
sample = pd.read_csv(DATA / 'sampleSubmission.csv', usecols=['Sl. No.'])
X, Q = train.to_numpy(float), test.to_numpy(float)
y = target_frame['Output'].to_numpy(float)
assert X.shape == (242, 64) and Q.shape == (200, 64) and y.shape == (242,)
assert train.columns.tolist() == test.columns.tolist()
assert np.isfinite(X).all() and np.isfinite(Q).all() and np.isfinite(y).all()
assert len(sample) == 200 and sample['Sl. No.'].is_unique
print('Training:', X.shape, 'Test:', Q.shape, 'Missing values:', int(train.isna().sum().sum()))
print('Training target range:', float(y.min()), float(y.max()))


Tested environment: {'python': '3.12', 'numpy': '2.3.5', 'pandas': '3.0.1', 'scipy': '1.18.1', 'scikit_learn': '1.9.1'}
Training: (242, 64) Test: (200, 64) Missing values: 0
Training target range: 25.0 341.0


## 2. Features and weighted ridge
I used the first 10 columns, all 64 columns, or a smaller set with added powers. I also tried different penalties for feature groups. Some training rows got more weight based on their target values.

I fitted scaling values and prediction limits using training rows only. There were no missing values.


In [3]:
RAW = [1,2,3,4,7,8,9,19,42,10,23]
def base_design(A,kind):
 Z=np.column_stack([A[:,RAW],A[:,8]**5,A[:,2]**5,A[:,5]**3,np.sign(A[:,8])*np.sqrt(abs(A[:,8]))])
 return np.column_stack([Z,A[:,21],A[:,1]**3]) if kind=='larger' else Z

def reference(A, b, Q, c):
    sc = StandardScaler()
    Z = sc.fit_transform(base_design(A, c['design']))
    V = sc.transform(base_design(Q, c['design']))
    z = (b-b.mean())/(b.std()+1e-12)
    low, high = (.5,.3) if c['design']=='larger' else (.2,.2)
    weights = 1+low*np.maximum(-z,0)**2+high*np.maximum(z,0)**2
    pred = Ridge(alpha=60.).fit(Z,b,sample_weight=weights).predict(V)
    return np.clip(pred,*np.quantile(b,[.1,.9]))


In [4]:
def weighted(A,b,Q,alpha=60.,low=.2,high=.2,clip=(.1,.9),extras=()):
    Z=base_design(A,'smaller');V=base_design(Q,'smaller')
    if extras:Z=np.column_stack([Z,A[:,extras]]);V=np.column_stack([V,Q[:,extras]])
    scaler=StandardScaler();Z=scaler.fit_transform(Z);V=scaler.transform(V)
    z=(b-b.mean())/(b.std()+1e-12);w=1+low*np.maximum(-z,0)**2+high*np.maximum(z,0)**2
    pred=Ridge(alpha=alpha).fit(Z,b,sample_weight=w).predict(V)
    return np.clip(pred,*np.quantile(b,clip)) if clip is not None else pred

def pca_forest_full(A,b,Q):
    pca=make_pipeline(StandardScaler(),PCA(n_components=4),RidgeCV(alphas=np.logspace(-3,3,100),cv=5,scoring='neg_mean_squared_error'))
    rf=RandomForestRegressor(n_estimators=500,max_depth=2,min_samples_leaf=6,max_features=1.,random_state=310,n_jobs=1)
    prediction=.655*pca.fit(A[:,:10],b).predict(Q[:,:10])+.345*rf.fit(A[:,:10],b).predict(Q[:,:10])
    return np.clip(prediction,b.min(),b.max())

def pca_forest_bag(A,b,Q):
    predictions=[]
    for seed in range(3000,3010):
        for tr,_ in KFold(5,shuffle=True,random_state=seed).split(A):
            # Original bagging clips only the final average, using full-fit bounds.
            pca=make_pipeline(StandardScaler(),PCA(n_components=4),RidgeCV(alphas=np.logspace(-3,3,100),cv=5,scoring='neg_mean_squared_error'))
            rf=RandomForestRegressor(n_estimators=500,max_depth=2,min_samples_leaf=6,max_features=1.,random_state=310,n_jobs=1)
            predictions.append(.655*pca.fit(A[tr,:10],b[tr]).predict(Q[:,:10])+.345*rf.fit(A[tr,:10],b[tr]).predict(Q[:,:10]))
    return np.clip(np.mean(predictions,axis=0),b.min(),b.max())

def supplied_group_ridge(A,b,Q,square,interaction):
    Z=A.copy();V=Q.copy()
    Z[:,10:19]*=square;V[:,10:19]*=square
    Z[:,19:64]*=interaction;V[:,19:64]*=interaction
    return np.clip(Ridge(alpha=.06).fit(Z,b).predict(V),b.min(),b.max())

def feature_multiplier_ridge(A,b,Q):
    Z=np.column_stack([base_design(A,'smaller'),A[:,21]])
    V=np.column_stack([base_design(Q,'smaller'),Q[:,21]])
    scaler=StandardScaler();Z=scaler.fit_transform(Z);V=scaler.transform(V)
    scale=np.r_[np.ones(11),[1.4,1.4,1.4,1.6],.6]
    Z*=scale;V*=scale
    z=(b-b.mean())/(b.std()+1e-12);weight=1+.3*np.maximum(-z,0)**2+.35*np.maximum(z,0)**2
    return np.clip(Ridge(alpha=60.).fit(Z,b,sample_weight=weight).predict(V),*np.quantile(b,[.1,.9]))


## 3. Model comparisons and the stack
I tried ridge, lasso, elastic net, PLS, splines, SVR, trees and kernel models. A stack combines their predictions with fitted weights. The weights below are fixed for the submitted model. The optional validation step fits the stack weights again inside each training fold.


In [5]:
def predict_base(
    name,
    X_train_raw,
    y_train,
    X_valid_raw,
):
    X10_train = X_train_raw[:, :10]
    X10_valid = X_valid_raw[:, :10]

    if name == "WeightedRidge":
        return weighted(
            X_train_raw,
            y_train,
            X_valid_raw,
        )

    if name == "Ridge10":
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=20.0),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "SplineRidge10":
        model = make_pipeline(
            StandardScaler(),
            SplineTransformer(
                n_knots=3,
                degree=2,
                include_bias=False,
            ),
            Ridge(alpha=1.0),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "PLS10":
        model = make_pipeline(
            StandardScaler(),
            PLSRegression(
                n_components=2
            ),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        ).ravel()

    if name == "SVR10":
        model = make_pipeline(
            StandardScaler(),
            SVR(
                C=30.0,
                gamma=0.03,
                epsilon=10.0,
            ),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "GBR10":
        model = GradientBoostingRegressor(
            n_estimators=90,
            learning_rate=0.03,
            max_depth=1,
            loss="huber",
            random_state=34,
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "RF10":
        model = RandomForestRegressor(
            n_estimators=110,
            max_depth=3,
            min_samples_leaf=6,
            max_features=0.8,
            random_state=34,
            n_jobs=1,
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "ExtraTrees10":
        model = ExtraTreesRegressor(
            n_estimators=110,
            max_depth=4,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=34,
            n_jobs=1,
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "Ridge64":
        model = make_pipeline(
            StandardScaler(),
            Ridge(alpha=60.0),
        )

        model.fit(
            X_train_raw,
            y_train,
        )

        return model.predict(
            X_valid_raw
        )

    if name == "Bayesian64":
        model = make_pipeline(
            StandardScaler(),
            BayesianRidge(),
        )

        model.fit(
            X_train_raw,
            y_train,
        )

        return model.predict(
            X_valid_raw
        )

    if name == "ARD64":
        model = make_pipeline(
            StandardScaler(),
            ARDRegression(
                max_iter=300
            ),
        )

        model.fit(
            X_train_raw,
            y_train,
        )

        return model.predict(
            X_valid_raw
        )

    if name == "Lasso64":
        model = make_pipeline(
            StandardScaler(),
            Lasso(
                alpha=1.0,
                max_iter=10000,
            ),
        )

        model.fit(
            X_train_raw,
            y_train,
        )

        return model.predict(
            X_valid_raw
        )

    if name == "ElasticNet64":
        model = make_pipeline(
            StandardScaler(),
            ElasticNet(
                alpha=0.10,
                l1_ratio=0.50,
                max_iter=10000,
            ),
        )

        model.fit(
            X_train_raw,
            y_train,
        )

        return model.predict(
            X_valid_raw
        )

    if name == "Huber10":
        model = make_pipeline(
            StandardScaler(),
            HuberRegressor(
                epsilon=1.35,
                alpha=1.0,
                max_iter=1000,
            ),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "OLS10":
        model = make_pipeline(
            StandardScaler(),
            LinearRegression(),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    if name == "KernelPoly10":
        model = make_pipeline(
            StandardScaler(),
            KernelRidge(
                alpha=10.0,
                kernel="polynomial",
                degree=2,
                gamma=0.10,
                coef0=1.0,
            ),
        )

        model.fit(
            X10_train,
            y_train,
        )

        return model.predict(
            X10_valid
        )

    raise ValueError(
        name
    )


In [6]:
STACK_NAMES = ['WeightedRidge', 'Ridge10', 'SplineRidge10', 'PLS10', 'SVR10', 'GBR10', 'RF10', 'ExtraTrees10', 'Ridge64', 'Bayesian64', 'ARD64', 'Lasso64', 'ElasticNet64', 'Huber10', 'OLS10', 'KernelPoly10']
STACK_WEIGHTS = np.array([1.3796929205792277, -1.432857523359001, 0.20738783185537774, 0.6096152829492533, -0.6615029203046363, -0.043315238777381526, 0.37374352751855566, -0.7241372076256305, 1.3300688230892916, -1.2243660914794494, -0.5310442860681537, 0.15658984766837392, -0.4756256020324847, 1.5903113900761419, -0.434561810977347, 0.8800010568878947])

TRAINING_LOCAL_STACK = False
STACK_CACHE = {}

def fit_stack_weights(A,b):
    key = hashlib.sha256(A.tobytes()+b.tobytes()).hexdigest()
    if key not in STACK_CACHE:
        rows, targets = [], []
        for seed in [14001,14002,14003,14004]:
            for tr,va in KFold(5,shuffle=True,random_state=seed).split(A):
                rows.append(np.column_stack([predict_base(n,A[tr],b[tr],A[va]) for n in STACK_NAMES]))
                targets.append(b[va])
        P, yy = np.vstack(rows), np.concatenate(targets)
        prior = np.zeros(16); prior[0] = 1.
        objective = lambda w: np.mean((yy-P@w)**2)+np.sum((w-prior)**2)
        fit = minimize(objective,prior,method='SLSQP',bounds=[(-2.,2.)]*16,
            constraints=[dict(type='eq',fun=lambda w:w.sum()-1.)],
            options=dict(maxiter=3000,ftol=1e-10))
        if not fit.success: raise RuntimeError(fit.message)
        STACK_CACHE[key] = fit.x
    return STACK_CACHE[key]

def stack_predict(A,b,Q):
    weights = fit_stack_weights(A,b) if TRAINING_LOCAL_STACK else STACK_WEIGHTS
    P = np.column_stack([predict_base(n,A,b,Q) for n in STACK_NAMES])
    return np.clip(P@weights,*np.quantile(b,[.05,.95]))


## 4. Other ways to change the predictions
One model used training errors to estimate which rows were harder to predict. Another used the inputs to change how four models were combined. Both use training labels only.


In [7]:
EXPERT_CACHE = {}
def powers(A):return np.column_stack([A[:,:10]**p for p in [1,2,3]])

def fit_experts(A,b):
 bounds=np.quantile(b,[.1,.9])
 sc=StandardScaler();Z=sc.fit_transform(powers(A));ridge=Ridge(alpha=17.78279410038923).fit(Z,b)
 ss=StandardScaler();U=ss.fit_transform(A[:,:10]);svr=SVR(C=300.,gamma=.01,epsilon=10.).fit(U,b)
 gbr=GradientBoostingRegressor(n_estimators=120,max_depth=1,learning_rate=.03,min_samples_leaf=15,random_state=78310).fit(A[:,:10],b)
 return [lambda Q:reference(A,b,Q,dict(method='reference',design='larger')),
  lambda Q:np.clip(ridge.predict(sc.transform(powers(Q))),*bounds),
  lambda Q:np.clip(svr.predict(ss.transform(Q[:,:10])),*bounds),
  lambda Q:np.clip(gbr.predict(Q[:,:10]),*bounds)]

def training_experts(A,b):
 key=hashlib.sha256(A.tobytes()+b.tobytes()).digest()
 if key not in EXPERT_CACHE:
  P=np.full((len(b),4),np.nan)
  for tr,va in KFold(3,shuffle=True,random_state=78310).split(A):
   experts=fit_experts(A[tr],b[tr]);P[va]=np.column_stack([expert(A[va]) for expert in experts])
  assert np.isfinite(P).all()
  EXPERT_CACHE[key]=(P,fit_experts(A,b))
 return EXPERT_CACHE[key]

def correction_design(P,H):
 differences=P[:,1:]-P[:,[0]]
 return np.column_stack([differences*H[:,j:j+1] for j in range(H.shape[1])])

def conditional_predict(A,b,Q,c):
 if c['method']=='reference':return reference(A,b,Q,c)
 P,experts=training_experts(A,b);Pq=np.column_stack([expert(Q) for expert in experts])
 if c['context']=='constant':H=np.ones((len(A),1));Hq=np.ones((len(Q),1))
 else:
  cs=StandardScaler();H=cs.fit_transform(A[:,[2,3,8]]);Hq=cs.transform(Q[:,[2,3,8]])
  H=np.column_stack([np.ones(len(A)),H]);Hq=np.column_stack([np.ones(len(Q)),Hq])
 D=correction_design(P,H);Dq=correction_design(Pq,Hq)
 ds=StandardScaler();D=ds.fit_transform(D);Dq=ds.transform(Dq)
 # Zero meta coefficients recover V27. Standardized differences make the
 # zero-centered prior comparable across heterogeneous expert predictions.
 coef=Ridge(alpha=c['alpha'],fit_intercept=False).fit(D,b-P[:,0]).coef_
 correction=np.clip(Dq@coef,-20.,20.)
 return np.clip(Pq[:,0]+correction,*np.quantile(b,[.1,.9]))


In [8]:
VARIANCE_CACHE = {}
def variance_ratio(A,b,c):
    key=hashlib.sha256(A.tobytes()+b.tobytes()+str(c['variance_alpha']).encode()).digest()
    if key not in VARIANCE_CACHE:
        oof=np.zeros(len(b))
        for tr,va in KFold(3,shuffle=True,random_state=464646).split(A):
            oof[va]=reference(A[tr],b[tr],A[va],dict(method='reference',design='larger'))
        residual2=(b-oof)**2
        target=residual2/(residual2.mean()+1e-12)+.1
        sc=StandardScaler();U=sc.fit_transform(np.column_stack([A[:,:10],A[:,:10]**2]))
        fit=GammaRegressor(alpha=c['variance_alpha'],max_iter=1000,tol=1e-8).fit(U,target)
        assert fit.n_iter_<1000,'Gamma variance fit did not converge within budget.'
        variance=fit.predict(U)
        VARIANCE_CACHE[key]=np.clip(variance.mean()/variance,.25,4.)
    return VARIANCE_CACHE[key]

def gamma_predict(A,b,Q,c):
    if c['method']=='reference':return reference(A,b,Q,c)
    sc=StandardScaler();Z=sc.fit_transform(base_design(A,'larger'));Zq=sc.transform(base_design(Q,'larger'))
    z=(b-b.mean())/(b.std()+1e-12)
    w=(1+.5*np.maximum(-z,0)**2+.3*np.maximum(z,0)**2)*variance_ratio(A,b,c)**c['strength']
    fit=Ridge(alpha=60.).fit(Z,b,sample_weight=w)
    return np.clip(fit.predict(Zq),*np.quantile(b,[.1,.9]))


In [9]:
RECIPES={
 'Weighted ridge':lambda A,b,Q:weighted(A,b,Q),
 'Alternate weighted ridge':lambda A,b,Q:weighted(A,b,Q,alpha=45.,low=.3,high=.3),
 'Ridge with three extra columns':lambda A,b,Q:weighted(A,b,Q,alpha=55.,low=.5,high=.25,clip=(.1,.925),extras=(62,21,29)),
 'Weighted ridge without limits':lambda A,b,Q:weighted(A,b,Q,clip=None),
 'Ridge on 64 columns':lambda A,b,Q:np.clip(RidgeCV(alphas=np.logspace(-5,2,300),cv=10,scoring='neg_mean_squared_error').fit(A,b).predict(Q),b.min(),b.max()),
 'PCA ridge and forest':pca_forest_full,
 'PCA and forest fold average':pca_forest_bag,
 'Conditional model blend':lambda A,b,Q:conditional_predict(A,b,Q,dict(method='conditional_signed',context='strong3',alpha=1000.)),
 'Ridge with variance weights':lambda A,b,Q:gamma_predict(A,b,Q,dict(method='gamma_variance',variance_alpha=.1,strength=1.)),
 'Three-group ridge':lambda A,b,Q:supplied_group_ridge(A,b,Q,.04,.15),
 'Two-group ridge':lambda A,b,Q:supplied_group_ridge(A,b,Q,.11,.11),
 'Ridge with feature multipliers':feature_multiplier_ridge,
}


## 5. Small prediction changes
I took the difference between two model predictions. I removed the part that repeated earlier changes, then limited the size of the new change. These steps use test inputs and predictions, not test labels.

Run predictions for all 200 test rows together because some steps use averages across the batch.


In [10]:
PROBE_PARAMETERS = {'stack_change_primary_fraction': 0.3730961448916969, 'reference_blend_plane_coefficients': [-0.8531260106587402, 0.346489941930961]}
def initial_blend(A,b,Q):
    return (.474368*RECIPES['Weighted ridge'](A,b,Q)
        +.67232*RECIPES['Alternate weighted ridge'](A,b,Q)
        +.310852*RECIPES['Ridge on 64 columns'](A,b,Q)
        -.0806*pca_forest_full(A,b,Q)-.37694*feature_multiplier_ridge(A,b,Q))

def probes(A,b,Q):
    parameter=PROBE_PARAMETERS
    p33=initial_blend(A,b,Q);p16=RECIPES['Alternate weighted ridge'](A,b,Q);raw=stack_predict(A,b,Q)
    p34=p33+parameter['stack_change_primary_fraction']*(raw-p33)
    a,c=parameter['reference_blend_plane_coefficients'];p38=p33+a*(p34-p33)+c*(p16-p33)
    B=np.column_stack([p34-p33,p16-p33])
    S=np.column_stack([A[:,:10],A[:,:10]**2]);T=np.column_stack([Q[:,:10],Q[:,:10]**2])
    scaler=MinMaxScaler();S=scaler.fit_transform(S);T=scaler.transform(T)
    polynomial=LinearRegression().fit(S,b).predict(T)
    d=polynomial-p38;d-=B@np.linalg.lstsq(B,d,rcond=None)[0]
    p39=p38+.5/(np.mean(abs(d))+1e-12)*d
    scaler=StandardScaler();Z=scaler.fit_transform(A[:,:10]);V=scaler.transform(Q[:,:10])
    def kernel(U,V):
        dist=np.maximum((U*U).sum(1)[:,None]+(V*V).sum(1)[None,:]-2*U@V.T,0)
        return np.exp(-.01*dist)
    coefficient=np.linalg.solve(kernel(Z,Z)+np.eye(len(b)),b-b.mean())
    candidate=b.mean()+kernel(V,Z)@coefficient
    B=np.column_stack([B,p39-p38]);d=candidate-p38;d-=B@np.linalg.lstsq(B,d,rcond=None)[0]
    scale=min(1./np.mean(abs(d)),5./np.max(abs(d)))
    p40=p38+scale*d
    return dict(initial_blend=p33,stack_change=p34,reference_blend=p38,polynomial_change=p39,kernel_change=p40)


In [11]:
def grouped_predict(A,b,Q,c):
    if c['method']=='reference':return reference(A,b,Q,c)
    scaler=StandardScaler();Z=scaler.fit_transform(base_design(A,'larger'));V=scaler.transform(base_design(Q,'larger'))
    penalty=np.ones(Z.shape[1]);penalty[[11,12,13,14]]=c['power'];penalty[[7,8,10,15]]=c['interaction']
    Z=Z/np.sqrt(penalty);V=V/np.sqrt(penalty)
    z=(b-b.mean())/(b.std()+1e-12);weight=1+c['tail']*np.maximum(-z,0)**2+.3*np.maximum(z,0)**2
    model=Ridge(alpha=c['alpha']).fit(Z,b,sample_weight=weight)
    return np.clip(model.predict(V),*np.quantile(b,[.1,.9]))


In [12]:
CENTER = dict(method='grouped',alpha=60.,power=.4,interaction=.8,tail=.5)
def training_setup(A,b,Q):
    # Residual weighting never sees outer validation targets.
    fitted=np.full(len(b),np.nan)
    for tr,va in KFold(3,shuffle=True,random_state=1359310).split(A):
        fitted[va]=grouped_predict(A[tr],b[tr],A[va],CENTER)
    residual=abs(b-fitted)
    scale=max(float(np.median(residual)/.67448975),1e-8)
    scaler=StandardScaler()
    Z=scaler.fit_transform(base_design(A,'larger'))
    V=scaler.transform(base_design(Q,'larger'))
    penalty=np.ones(Z.shape[1]);penalty[[11,12,13,14]]=.4;penalty[[7,8,10,15]]=.8
    Z/=np.sqrt(penalty);V/=np.sqrt(penalty)
    z=(b-b.mean())/(b.std()+1e-12)
    weights=1+.5*np.maximum(-z,0)**2+.3*np.maximum(z,0)**2
    return Z,V,residual,scale,weights

def robust_predict(setup,b,c):
    Z,V,residual,scale,weights=setup
    multiplier=np.maximum(c['floor'],np.minimum(1.,c['cutoff']*scale/(residual+1e-12)))
    pred=Ridge(alpha=60.).fit(Z,b,sample_weight=weights*multiplier).predict(V)
    return np.clip(pred,*np.quantile(b,[.1,.9]))

def correction(direction,known,strength):
    d=direction-known@np.linalg.lstsq(known,direction,rcond=1e-10)[0]
    scale=min(strength,2./max(float(np.max(abs(d))),1e-12))
    return scale*d

def penalty_delta(alternate, center, known):
    d = alternate-center
    d -= known*float(d@known/(known@known+1e-20))
    return min(.25,.5/max(float(np.max(abs(d))),1e-12))*d

def extra_components(A,b,Q):
    center05 = grouped_predict(A,b,Q,dict(method='grouped',alpha=60.,power=.5,interaction=.7,tail=.5))
    old = center05-reference(A,b,Q,dict(method='reference',design='larger'))
    center045 = grouped_predict(A,b,Q,dict(method='grouped',alpha=60.,power=.45,interaction=.7,tail=.5))
    first = penalty_delta(center045,center05,old)
    center = grouped_predict(A,b,Q,CENTER)
    second = penalty_delta(center,center045,old)
    alternative = grouped_predict(A,b,Q,dict(method='grouped',alpha=65.,power=.4,interaction=.8,tail=.65))
    d = alternative-center
    known = np.column_stack([old,first+second])
    d -= known@np.linalg.lstsq(known,d,rcond=1e-10)[0]
    one = min(.25,.5/max(float(np.max(abs(d))),1e-12))*d
    setup = training_setup(A,b,Q)
    direction = robust_predict(setup,b,dict(cutoff=1.5,floor=.25))-center
    power = correction(direction,np.column_stack([old,first+second,one]),1.)
    return np.column_stack([old,first,second,one,power])


## 6. Final blend weights
Multiply each model prediction or small change by its weight, then add them. A negative weight subtracts that part. The code keeps the exact weights needed to recreate the scored result. The displayed table rounds them for readability.


In [13]:
COMPONENT_NAMES = ['Weighted ridge without limits', 'PCA and forest fold average', 'Stack of 16 models', 'Ridge on 64 columns', 'Ridge with three extra columns', 'Reference prediction blend', 'Three-group ridge', 'Ridge with variance weights', 'Conditional model blend', 'Ridge with feature multipliers', 'Ridge with tighter limits', 'Two-group ridge', 'Weighted ridge', 'PCA ridge and forest', 'Polynomial prediction change', 'Kernel prediction change', 'constant', 'group_change', 'first_penalty_change', 'second_penalty_change', 'tail_alpha_change', 'residual_weight_change']
FINAL_WEIGHTS = np.array([-0.00708648821922271, 2.093296266022348, 0.7405607884008948, -0.08168762624696818, 0.35766959878948124, -18.51485986684993, -0.5305472976426937, -0.23939019162084185, 0.03188612676660025, 0.0737647840312497, -0.8028723643275214, 0.40195970895445265, -1.2881849361833348, -3.0561676488391054, 7.312451863431066, 14.509207283533593, -9.25829234161203e-12, -1.0918467272645038, 6.016913557987363, 6.016913557646539, -5.771277248753549, 6.362512413991007])

def final_components(A,b,Q):
    values = {name:fit(A,b,Q) for name,fit in RECIPES.items() if name!='Alternate weighted ridge'}
    values['Ridge with tighter limits'] = np.clip(values['Weighted ridge'],*np.quantile(b,[.12,.88]))
    values['Stack of 16 models'] = stack_predict(A,b,Q)
    probe_values = probes(A,b,Q)
    values.update({'Reference prediction blend':probe_values['reference_blend'],
        'Polynomial prediction change':probe_values['polynomial_change'],
        'Kernel prediction change':probe_values['kernel_change'],'constant':np.ones(len(Q))})
    return np.column_stack([values[n] for n in COMPONENT_NAMES[:17]]+[extra_components(A,b,Q)])

def predict_final(A,b,Q):
    return final_components(A,b,Q)@FINAL_WEIGHTS

print(pd.DataFrame({'component':COMPONENT_NAMES,'weight':np.round(FINAL_WEIGHTS,5)}).to_string(index=False))


                     component    weight
 Weighted ridge without limits  -0.00709
   PCA and forest fold average   2.09330
            Stack of 16 models   0.74056
           Ridge on 64 columns  -0.08169
Ridge with three extra columns   0.35767
    Reference prediction blend -18.51486
             Three-group ridge  -0.53055
   Ridge with variance weights  -0.23939
       Conditional model blend   0.03189
Ridge with feature multipliers   0.07376
     Ridge with tighter limits  -0.80287
               Two-group ridge   0.40196
                Weighted ridge  -1.28818
          PCA ridge and forest  -3.05617
  Polynomial prediction change   7.31245
      Kernel prediction change  14.50921
                      constant  -0.00000
                  group_change  -1.09185
          first_penalty_change   6.01691
         second_penalty_change   6.01691
             tail_alpha_change  -5.77128
        residual_weight_change   6.36251


## 7. Make the submission
The notebook writes a new file in the reproduced folder. It keeps the original scored file unchanged. If that original file is beside the notebook, the code checks that the predictions match. It only loads that file for the check.


In [14]:
with threadpool_limits(limits=1):
    predictions = predict_final(X,y,Q)
assert np.isfinite(predictions).all() and len(predictions)==200
submission = pd.DataFrame({'Sl. No.':sample['Sl. No.'],'Output':predictions})
out = Path('reproduced'); out.mkdir(exist_ok=True)
output = out / 'submission.csv'
submission.to_csv(output,index=False)
assert submission.columns.tolist()==['Sl. No.','Output']
original = next((p for p in [Path('submissions/final_submission.csv'), Path('../submissions/final_submission.csv')] if p.is_file()), Path('missing_original'))
if original.is_file():
    saved = pd.read_csv(original)
    assert submission['Sl. No.'].equals(saved['Sl. No.'])
    difference = float(np.max(abs(predictions-saved['Output'].to_numpy())))
    print('Maximum difference from the scored submission:', difference)
    assert difference < 1e-6
print('Created:', output.resolve())
print(submission.head().to_string(index=False))


Maximum difference from the scored submission: 1.3061196568742162e-10
Created: reproduced/submission.csv
 Sl. No.     Output
       1  83.561254
       2  90.911101
       3 102.268374
       4 118.911634
       5 126.775295


## 8. Scores and validation
These are actual public Kaggle scores, rounded to two decimal places. They are not training or local validation scores.

A local improvement did not always help on Kaggle. I used both local checks and public scores during the search, so neither is a completely fresh test of the final choices.


In [15]:
PUBLIC_RESULTS = [('Ridge on 64 columns',3190.59729),('Weighted ridge with added features',3051.87090),('Reference blend with small changes',3013.39848),('Blend with feature penalty changes',2965.97189),('Blend with tail and alpha change',2966.16381),('Blend with residual weights',2959.82725),('Earlier signed model blend',2942.52151),('Final signed model blend',2930.80324)]
results = pd.DataFrame(PUBLIC_RESULTS,columns=['model','public_MSE'])
print(results.round(2).to_string(index=False))
print('Lower MSE is better.')


                             model  public_MSE
               Ridge on 64 columns     3190.60
Weighted ridge with added features     3051.87
Reference blend with small changes     3013.40
Blend with feature penalty changes     2965.97
  Blend with tail and alpha change     2966.16
       Blend with residual weights     2959.83
        Earlier signed model blend     2942.52
          Final signed model blend     2930.80
Lower MSE is better.


### Optional final validation
Leave RUN_FINAL_CV=False for the normal run. Set it to True to repeat the five fold local check. It takes longer because the stack weights are fitted inside each training fold. The final features and blend weights stay fixed, so this still is not a fresh test of the whole search.


In [16]:
RUN_FINAL_CV = False
if RUN_FINAL_CV:
    TRAINING_LOCAL_STACK = True
    held_out = np.full(len(y),np.nan)
    try:
        with threadpool_limits(limits=1):
            for fold,(tr,va) in enumerate(KFold(5,shuffle=True,random_state=1399310).split(X),1):
                held_out[va] = predict_final(X[tr],y[tr],X[va])
                print('Completed validation fold',fold)
        print('Training-local diagnostic MSE:',mean_squared_error(y,held_out))
    finally:
        TRAINING_LOCAL_STACK = False
else:
    print('Optional final-model CV skipped. Reported Kaggle MSE: 2930.80324.')


Optional final-model CV skipped. Reported Kaggle MSE: 2930.80324.


## 9. What I learned
A bigger model did not always do better. Ridge with careful feature choices was a useful start. Small changes to feature penalties and training weights helped. Some changes looked good locally but did worse on Kaggle.

My final public MSE was **2930.80**. The exact score was **2930.80324**. The local five fold check was **3483.96**. I would keep a separate validation set and use fewer public checks if I did this again.
